# MAX5885 PL UART — MMIO 16550 Control

PL AXI UART 16550 on W9(RX)/W10(TX), 115200 8E1, MMIO direct register access.

**No kernel driver needed.** All registers operated via `/dev/mem`.

## Deploy these files to Jupyter directory
- `base_add.bit` + `base_add.hwh`
- `pynqz1_diansai2_max5885.py`

## Usage
Run cells in order: **Cell 1 → Cell 2 → Cell 3**

In [ ]:
# Cell 1: load overlay, init 16550 + MAX5885, define all helpers.
from time import sleep, time
import os
from pynq import Overlay, MMIO
from pynqz1_diansai2_max5885 import MAX5885Signal

# ---- addresses from base_add.hwh MEMORYMAP ----
UART_BASE    = 0x43C00000
LED_BASE     = 0x40000000
MAX5885_BASE = 0x40001000
AXI_CLK_HZ   = 125000000
BAUD         = 115200

# ---- load overlay ----
overlay = Overlay('base_add.bit')
assert 'uart_0' in overlay.ip_dict, 'uart_0 missing from overlay!'
print('Overlay loaded:', list(overlay.ip_dict.keys()))

# ---- MMIO + DAC ----
uart = MMIO(UART_BASE, 0x10000)
led  = MMIO(LED_BASE, 0x1000)
dac  = MAX5885Signal(MAX5885_BASE)
print('DAC v=0x%08X rate=%d' % (dac.read(0x38), dac.read(0x3C)))

# ============================================================
#  16550 register init (115200 8E1)
# ============================================================
RBR_THR_DLL, IER_DLM, FCR_IIR = 0x00, 0x04, 0x08
LCR, MCR, LSR, MSR, SCR = 0x0C, 0x10, 0x14, 0x18, 0x1C
LSR_RX_READY, LSR_TX_EMPTY = 0x01, 0x20

divisor = round(AXI_CLK_HZ / (16 * BAUD))
uart.write(LCR, 0x80);                       # DLAB=1
uart.write(RBR_THR_DLL, divisor & 0xFF);      # DLL
uart.write(IER_DLM, (divisor >> 8) & 0xFF);  # DLM
uart.write(LCR, 0x1B);                       # 8E1, DLAB=0
uart.write(FCR_IIR, 0x07);                   # FIFO on + clear
uart.write(IER_DLM, 0x00);                   # no interrupts
print('16550 init: divisor=%d baud=%.0f' % (divisor, AXI_CLK_HZ/(16*divisor)))

# ============================================================
#  UART helpers
# ============================================================
def uart_tx_byte(b):
    while not (uart.read(LSR) & LSR_TX_EMPTY): pass
    uart.write(RBR_THR_DLL, b & 0xFF)

def uart_rx_byte(timeout_ms=200):
    dl = time() + timeout_ms / 1000
    while time() < dl:
        if uart.read(LSR) & LSR_RX_READY:
            return uart.read(RBR_THR_DLL) & 0xFF
    return None

def uart_send(line):
    for b in (str(line).strip('\r\n') + '\r\n').encode('ascii'):
        uart_tx_byte(b)
    print('TX:', line)

def uart_read_line(timeout_s=1.0):
    dl, raw = time() + timeout_s, bytearray()
    while time() < dl:
        ch = uart_rx_byte(100)
        if ch == 0x0A:
            line = raw.rstrip(b'\r').decode('ascii', errors='replace')
            print('RX:', line)
            return line
        if ch is not None and len(raw) < 255:
            raw.append(ch)
    return None

# ============================================================
#  LED pulse
# ============================================================
class LedPulse:
    def __init__(self):
        self._restore = None
        self._dl = 0.0
    def pulse(self, dur=0.5):
        if self._restore is None:
            self._restore = led.read(0x08)
        led.write(0x00, 0)
        led.write(0x08, self._restore | 0x2)
        self._dl = time() + dur
    def update(self):
        if self._restore is not None and time() >= self._dl:
            led.write(0x08, self._restore)
            self._restore = None

lp = LedPulse()
print('Ready. UART W9(RX)/W10(TX) 115200 8E1')

In [ ]:
# Cell 2: WCFG parser + apply (STM32 14-field format).
VALID_OUT = {'SD','SM','SOUT','DC','SQUARE','MOD_SINE'}

def handle_wcfg(line):
    """Parse STM32 WCFG line and apply to MAX5885. Returns True on success."""
    if not line.upper().startswith('WCFG,'):
        return False
    try:
        f = [x.strip() for x in line.split(',')]
        if len(f) != 14:
            raise ValueError('need 14 fields, got %d' % len(f))
        # WCFG,carrier,mod,square,mode,sd_vrms,am_depth,sm_delay,sm_phase,
        #       sm_atten,dac_a_gain,dac_b_gain,out_a,out_b
        cfg = dict(
            carrier_hz       = int(f[1]),
            mod_hz           = int(f[2]),
            square_hz        = int(f[3]),
            mode             = f[4].upper(),
            sd_vrms          = float(f[5]),
            am_depth_percent = float(f[6]),
            sm_delay_ns      = float(f[7]),
            sm_phase_deg     = float(f[8]),
            sm_atten_db      = float(f[9]),
            dac_a_gain       = float(f[10]),
            dac_b_gain       = float(f[11]),
            out_a            = f[12].upper(),
            out_b            = f[13].upper(),
            sd_phase_deg     = 0.0,
        )
        if cfg['mode'] not in ('CW','AM'):
            raise ValueError('mode: CW/AM')
        if cfg['out_a'] not in VALID_OUT or cfg['out_b'] not in VALID_OUT:
            raise ValueError('output: ' + ','.join(sorted(VALID_OUT)))

        actual = dac.configure_wireless(**cfg)
        lp.pulse(0.5)
        uart_send('STATUS,PYNQ APPLIED %dMHz %s' % (
            round(cfg['carrier_hz']/1e6), cfg['mode']))
        uart_send('LOG,A=%s B=%s SD=%dmV' % (
            cfg['out_a'], cfg['out_b'], round(cfg['sd_vrms']*1000)))
        print('OK: carrier=%.3fMHz' % actual['actual_carrier_hz']/1e6)
        return True
    except Exception as e:
        uart_send('ERR,%s' % str(e)[:80])
        print('ERR:', e)
        return False

print('WCFG parser ready. Fields: carrier,mod,square,mode,sd,am_depth,sm_delay,sm_phase,sm_atten,gain_a,gain_b,out_a,out_b')

In [ ]:
# Cell 3: link test + auto-start continuous listener.
import time

# drain stale bytes
while uart_rx_byte(50) is not None:
    pass

uart_send('STATUS,PYNQ MMIO 16550 READY')
uart_send('TEST,UART TX OK 8E1')
uart_send('LOG,PL UART W9/W10 115200 8E1')

print('=== Link test: waiting for STM32 WCFG (10s) ===')
got = False
dl = time.time() + 10
while time.time() < dl and not got:
    lp.update()
    line = uart_read_line(0.25)
    if line and line.upper().startswith('WCFG,'):
        uart_send('TEST,UART RX OK 8E1')
        got = handle_wcfg(line)

if not got:
    print('No WCFG. Check wiring: STM32 TX->W9, RX<-W10, GND.')
else:
    print('=== Link OK. Starting continuous listener (interrupt to stop) ===')
    try:
        while True:
            lp.update()
            line = uart_read_line(0.3)
            if line:
                handle_wcfg(line)
    except KeyboardInterrupt:
        print('Stopped.')